# InsureWise AI - Production Model Sanity & Robustness Verification
**Notebook:** `11_model_sanity_test.ipynb`  
**Project:** InsureWise AI - End-to-End Machine Learning Service  
**Objective:** Perform comprehensive validation and sanity testing on the saved production model pipeline (`insurewise_model.joblib`). Verify model loading, direct raw input prediction support, numeric scalar outputs, reasonable price boundary bounds, profile-based risk sensitivity, missing/unknown input resilience, and zero requirement for manual external preprocessing.

---

## Table of Contents
1. [Environment Setup & Class Registration](#1.-Environment-Setup-&-Class-Registration)
2. [Verification 1: Production Pipeline Artifact Loading](#2.-Verification-1:-Production-Pipeline-Artifact-Loading)
3. [Verification 2, 3 & 4: Direct Raw Input, Numeric Output & Range Bounds](#3.-Verification-2,-3-&-4:-Direct-Raw-Input,-Numeric-Output-&-Range-Bounds)
4. [Verification 5: Multi-Profile Comparative Risk Sensitivity](#4.-Verification-5:-Multi-Profile-Comparative-Risk-Sensitivity)
5. [Verification 6: Missing & Invalid Input Resilience](#5.-Verification-6:-Missing-&-Invalid-Input-Resilience)
6. [Verification 7: Zero External Preprocessing Guarantee](#6.-Verification-7:-Zero-External-Preprocessing-Guarantee)
7. [Bulk Batch Prediction Integrity Test](#7.-Bulk-Batch-Prediction-Integrity-Test)
8. [Requirements Compliance Verification Matrix](#8.-Requirements-Compliance-Verification-Matrix)

---
> **Strict Non-Retraining Policy:**
> - This notebook **does not retrain** the model.
> - It evaluates the exact persisted production artifact `models/production/insurewise_model.joblib` as a black-box service.

In [1]:
import os
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin

warnings.filterwarnings('ignore')

# Global Styling & Setup
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

class BloodPressureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, bp_col='blood_pressure'):
        self.bp_col = bp_col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            X_out = X.copy()
        else:
            X_out = pd.DataFrame(X).copy()

        if self.bp_col in X_out.columns:
            split_bp = X_out[self.bp_col].astype(str).str.split('/', expand=True)
            if split_bp.shape[1] >= 2:
                X_out['systolic_bp'] = pd.to_numeric(split_bp[0], errors='coerce').astype('float64')
                X_out['diastolic_bp'] = pd.to_numeric(split_bp[1], errors='coerce').astype('float64')
            elif split_bp.shape[1] == 1:
                X_out['systolic_bp'] = pd.to_numeric(split_bp[0], errors='coerce').astype('float64')
                X_out['diastolic_bp'] = np.nan
            else:
                X_out['systolic_bp'] = np.nan
                X_out['diastolic_bp'] = np.nan
            X_out = X_out.drop(columns=[self.bp_col])
        return X_out

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return None
        feature_names = [f for f in input_features if f != self.bp_col]
        feature_names.extend(['systolic_bp', 'diastolic_bp'])
        return np.array(feature_names)

print("Environment setup and BloodPressureExtractor class definition completed.")


Environment setup and BloodPressureExtractor class definition completed.


## 2. Verification 1: Production Pipeline Artifact Loading

We load the saved production pipeline artifact from `models/production/insurewise_model.joblib` and verify its object class and step configuration.

In [2]:
model_path = '../models/production/insurewise_model.joblib'
if not os.path.exists(model_path):
    model_path = r'c:\Users\Thiru kumaran v\Downloads\ML_project\Ml_Service\models\production\insurewise_model.joblib'

production_pipeline = joblib.load(model_path)

print("==========================================================================================================")
print("                       PRODUCTION MODEL ARTIFACT LOAD VERIFICATION")
print("==========================================================================================================")
print(f"Artifact Path: {model_path}")
print(f"Loaded Object Type: {type(production_pipeline)}")
print("Pipeline Steps:")
for step_name, step_obj in production_pipeline.named_steps.items():
    print(f"  - [{step_name}]: {type(step_obj).__name__}")

print("\nModel Loading Verification: VERIFIED SUCCESSFUL!")


                       PRODUCTION MODEL ARTIFACT LOAD VERIFICATION
Artifact Path: c:\Users\Thiru kumaran v\Downloads\ML_project\Ml_Service\models\production\insurewise_model.joblib
Loaded Object Type: <class 'sklearn.pipeline.Pipeline'>
Pipeline Steps:
  - [bp_extractor]: BloodPressureExtractor
  - [preprocessor]: ColumnTransformer
  - [regressor]: XGBRegressor

Model Loading Verification: VERIFIED SUCCESSFUL!


## 3. Verification 2, 3 & 4: Direct Raw Input, Numeric Output & Range Bounds

We pass an un-preprocessed raw customer record (including string blood pressure `"120/80"`) into `production_pipeline.predict()` and verify:
1. Direct raw input acceptance.
2. Numeric scalar prediction output (`float`).
3. Reasonable price boundary bounds ($\$1,500 \le \text{pred} \le \$55,000$).

In [3]:
raw_single_customer = pd.DataFrame([{
    'age': 30,
    'gender': 'Male',
    'bmi': 23.5,
    'children': 1,
    'smoker': 'No',
    'region': 'Southwest',
    'occupation': 'Teacher',
    'annual_income_usd': 52000,
    'exercise_level': 'High',
    'chronic_diseases': 0,
    'doctor_visits_per_year': 3,
    'hospitalizations_last_year': 0,
    'alcohol_consumption_per_week': 2,
    'insurance_plan': 'Standard',
    'blood_pressure': '120/80',
    'diabetes': 'No',
    'cholesterol': 185,
    'sleep_hours': 7.5,
    'stress_level': 4.0,
    'marital_status': 'Single'
}])

pred_val = production_pipeline.predict(raw_single_customer)[0]

print("==========================================================================================================")
print("                   RAW INPUT PREDICTION & RANGE BOUNDS VERIFICATION")
print("==========================================================================================================")
print(f"Raw Input Blood Pressure: '{raw_single_customer.iloc[0]['blood_pressure']}'")
print(f"Predicted Premium Output: ${float(pred_val):,.2f}")
print(f"Output Data Type:         {type(pred_val)}")

# Assertions
assert isinstance(pred_val, (float, np.floating, np.number)), "Assertion Failed: Prediction must be numeric!"
assert 1500.0 <= pred_val <= 55000.0, f"Assertion Failed: Prediction ${pred_val:,.2f} is outside realistic range [$1,500, $55,000]!"

print("Direct Raw Input Support:  VERIFIED")
print("Numeric Prediction Type:   VERIFIED")
print("Realistic Range Boundary:  VERIFIED")


                   RAW INPUT PREDICTION & RANGE BOUNDS VERIFICATION
Raw Input Blood Pressure: '120/80'
Predicted Premium Output: $7,098.80
Output Data Type:         <class 'numpy.float32'>
Direct Raw Input Support:  VERIFIED
Numeric Prediction Type:   VERIFIED
Realistic Range Boundary:  VERIFIED


## 4. Verification 5: Multi-Profile Comparative Risk Sensitivity

We test 4 realistic, distinct patient profiles to verify that the pipeline produces sensible, monotonic premium adjustments aligned with risk factors:
- **Profile A (Young Healthy Non-Smoker):** 22yo, Non-smoker, BMI 21.5, 0 chronic, Basic plan.
- **Profile B (Middle-Aged Moderate Risk):** 42yo, Non-smoker, BMI 27.5, 1 chronic, Standard plan.
- **Profile C (Senior High-Risk Smoker):** 61yo, Smoker, BMI 34.2, 3 chronic, Premium plan, 2 hospitalizations.
- **Profile D (Young High-Risk Smoker):** 25yo, Smoker, BMI 31.0, 1 chronic, Standard plan.

In [4]:
profiles = [
    {
        'Profile_ID': 'Profile A (Young Healthy Non-Smoker)',
        'data': {
            'age': 22, 'gender': 'Female', 'bmi': 21.5, 'children': 0, 'smoker': 'No',
            'region': 'Southwest', 'occupation': 'Student', 'annual_income_usd': 25000,
            'exercise_level': 'High', 'chronic_diseases': 0, 'doctor_visits_per_year': 2,
            'hospitalizations_last_year': 0, 'alcohol_consumption_per_week': 1,
            'insurance_plan': 'Basic', 'blood_pressure': '115/75', 'diabetes': 'No',
            'cholesterol': 170, 'sleep_hours': 8.0, 'stress_level': 3.0, 'marital_status': 'Single'
        }
    },
    {
        'Profile_ID': 'Profile B (Middle-Aged Moderate Risk)',
        'data': {
            'age': 42, 'gender': 'Male', 'bmi': 27.5, 'children': 2, 'smoker': 'No',
            'region': 'Northwest', 'occupation': 'Manager', 'annual_income_usd': 75000,
            'exercise_level': 'Medium', 'chronic_diseases': 1, 'doctor_visits_per_year': 5,
            'hospitalizations_last_year': 0, 'alcohol_consumption_per_week': 4,
            'insurance_plan': 'Standard', 'blood_pressure': '128/82', 'diabetes': 'No',
            'cholesterol': 205, 'sleep_hours': 7.0, 'stress_level': 5.5, 'marital_status': 'Married'
        }
    },
    {
        'Profile_ID': 'Profile D (Young High-Risk Smoker)',
        'data': {
            'age': 25, 'gender': 'Male', 'bmi': 31.0, 'children': 0, 'smoker': 'Yes',
            'region': 'Southeast', 'occupation': 'Construction Worker', 'annual_income_usd': 45000,
            'exercise_level': 'Low', 'chronic_diseases': 1, 'doctor_visits_per_year': 6,
            'hospitalizations_last_year': 0, 'alcohol_consumption_per_week': 7,
            'insurance_plan': 'Standard', 'blood_pressure': '132/85', 'diabetes': 'No',
            'cholesterol': 220, 'sleep_hours': 6.0, 'stress_level': 6.5, 'marital_status': 'Single'
        }
    },
    {
        'Profile_ID': 'Profile C (Senior High-Risk Smoker)',
        'data': {
            'age': 61, 'gender': 'Male', 'bmi': 34.2, 'children': 2, 'smoker': 'Yes',
            'region': 'Northeast', 'occupation': 'Engineer', 'annual_income_usd': 110000,
            'exercise_level': 'Low', 'chronic_diseases': 3, 'doctor_visits_per_year': 14,
            'hospitalizations_last_year': 2, 'alcohol_consumption_per_week': 8,
            'insurance_plan': 'Premium', 'blood_pressure': '145/95', 'diabetes': 'Yes',
            'cholesterol': 245, 'sleep_hours': 5.5, 'stress_level': 8.5, 'marital_status': 'Married'
        }
    }
]

profile_results = []
for p in profiles:
    df_p = pd.DataFrame([p['data']])
    pred = float(production_pipeline.predict(df_p)[0])
    profile_results.append({
        'Profile': p['Profile_ID'],
        'Age': p['data']['age'],
        'Smoker': p['data']['smoker'],
        'BMI': p['data']['bmi'],
        'Chronic Diseases': p['data']['chronic_diseases'],
        'Predicted Premium ($)': pred
    })

df_profile_res = pd.DataFrame(profile_results)

print("==========================================================================================================")
print("                   MULTI-PROFILE COMPARATIVE PREMIUM SANITY MATRIX")
print("==========================================================================================================")
print(df_profile_res.to_string(index=False))

# Logical Order Verification: Young Healthy < Middle-Aged < Young Smoker < Senior High-Risk Smoker
pred_A = df_profile_res.loc[0, 'Predicted Premium ($)']
pred_B = df_profile_res.loc[1, 'Predicted Premium ($)']
pred_D = df_profile_res.loc[2, 'Predicted Premium ($)']
pred_C = df_profile_res.loc[3, 'Predicted Premium ($)']

assert pred_A < pred_B, "Sanity Failure: Profile A should be less than Profile B!"
assert pred_B < pred_D, "Sanity Failure: Profile B (Non-smoker) should be less than Profile D (Smoker)!"
assert pred_D < pred_C, "Sanity Failure: Profile D should be less than Profile C (Senior Multi-disease Smoker)!"

print("\nLogical Profile Monotonicity: VERIFIED SUCCESSFUL (Profile A < Profile B < Profile D < Profile C)")


                   MULTI-PROFILE COMPARATIVE PREMIUM SANITY MATRIX
                              Profile  Age Smoker     BMI  Chronic Diseases  Predicted Premium ($)
 Profile A (Young Healthy Non-Smoker)   22     No 21.5000                 0              5172.3481
Profile B (Middle-Aged Moderate Risk)   42     No 27.5000                 1             14917.7197
   Profile D (Young High-Risk Smoker)   25    Yes 31.0000                 1             27156.3418
  Profile C (Senior High-Risk Smoker)   61    Yes 34.2000                 3             45195.3086

Logical Profile Monotonicity: VERIFIED SUCCESSFUL (Profile A < Profile B < Profile D < Profile C)


## 5. Verification 6: Missing & Invalid Input Resilience

We test how the production pipeline handles missing numeric values, unparseable/missing blood pressure, unknown nominal categories, and unknown ordinal categories:
1. Missing blood pressure (`"blood_pressure": None`) -> `BloodPressureExtractor` extracts NaNs, `SimpleImputer` imputes medians.
2. Missing numeric values (`"bmi": None`, `"annual_income_usd": np.nan`) -> `SimpleImputer(strategy='median')` imputes medians.
3. Unknown nominal categories (`"region": "Mars"`, `"occupation": "Astronaut"`) -> `OneHotEncoder(handle_unknown='ignore')` encodes unknown categories as all zeros.
4. Unknown ordinal categories (`"exercise_level": "Extreme"`) -> `OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)` assigns `-1` without throwing an exception.

In [5]:
raw_edge_case = pd.DataFrame([{
    'age': 35,
    'gender': 'Female',
    'bmi': np.nan,  # Missing numeric
    'children': 1,
    'smoker': 'No',
    'region': 'Mars',  # Unknown nominal category
    'occupation': 'Astronaut',  # Unknown nominal category
    'annual_income_usd': np.nan,  # Missing numeric
    'exercise_level': 'Extreme',  # Unknown ordinal category
    'chronic_diseases': 0,
    'doctor_visits_per_year': 3,
    'hospitalizations_last_year': 0,
    'alcohol_consumption_per_week': 2,
    'insurance_plan': 'Standard',
    'blood_pressure': None,  # Missing blood pressure string
    'diabetes': 'No',
    'cholesterol': 185,
    'sleep_hours': 7.5,
    'stress_level': 4.0,
    'marital_status': 'Single'
}])

print("==========================================================================================================")
print("                   MISSING & INVALID INPUT RESILIENCE TEST")
print("==========================================================================================================")
print("Input Edge Case Record:")
print("  - blood_pressure: None")
print("  - bmi: NaN")
print("  - region: 'Mars' (Unknown)")
print("  - occupation: 'Astronaut' (Unknown)")
print("  - exercise_level: 'Extreme' (Unknown)")

pred_edge = float(production_pipeline.predict(raw_edge_case)[0])

print("----------------------------------------------------------------------------------------------------------")
print(f"Edge Case Prediction Result: ${pred_edge:,.2f}")
assert isinstance(pred_edge, float) and not np.isnan(pred_edge), "Assertion Failed: Edge case prediction must be a valid float!"
print("Missing / Invalid Input Resilience: VERIFIED SUCCESSFUL!")


                   MISSING & INVALID INPUT RESILIENCE TEST
Input Edge Case Record:
  - blood_pressure: None
  - bmi: NaN
  - region: 'Mars' (Unknown)
  - occupation: 'Astronaut' (Unknown)
  - exercise_level: 'Extreme' (Unknown)
----------------------------------------------------------------------------------------------------------
Edge Case Prediction Result: $10,572.31
Missing / Invalid Input Resilience: VERIFIED SUCCESSFUL!


## 6. Verification 7: Zero External Preprocessing Guarantee

We verify that no manual feature extraction, string parsing, scaling, or one-hot encoding code needs to be executed outside the pipeline. The saved artifact `insurewise_model.joblib` operates as a complete self-contained prediction engine.

In [6]:
# Minimal inference code required in FastAPI / Production:
def predict_insurance_cost(raw_dict: dict) -> float:
    input_df = pd.DataFrame([raw_dict])
    prediction = float(production_pipeline.predict(input_df)[0])
    return round(prediction, 2)

test_dict = {
    'age': 38, 'gender': 'Female', 'bmi': 26.2, 'children': 1, 'smoker': 'No',
    'region': 'Southeast', 'occupation': 'Nurse', 'annual_income_usd': 68000,
    'exercise_level': 'Medium', 'chronic_diseases': 0, 'doctor_visits_per_year': 4,
    'hospitalizations_last_year': 0, 'alcohol_consumption_per_week': 3,
    'insurance_plan': 'Standard', 'blood_pressure': '122/78', 'diabetes': 'No',
    'cholesterol': 195, 'sleep_hours': 7.2, 'stress_level': 4.5, 'marital_status': 'Married'
}

result_premium = predict_insurance_cost(test_dict)
print("==========================================================================================================")
print("                   ZERO EXTERNAL PREPROCESSING VERIFICATION")
print("==========================================================================================================")
print(f"Inference Function Call Result: ${result_premium:,.2f}")
print("Zero External Preprocessing Guarantee: VERIFIED SUCCESSFUL!")


                   ZERO EXTERNAL PREPROCESSING VERIFICATION
Inference Function Call Result: $10,726.71
Zero External Preprocessing Guarantee: VERIFIED SUCCESSFUL!


## 7. Bulk Batch Prediction Integrity Test

We load a 100-sample batch from `medical_insurance_cost_dataset_enhanced.csv` and execute batch prediction directly through `production_pipeline.predict()` to evaluate inference throughput and prediction statistics.

In [7]:
data_path = '../data/raw/medical_insurance_cost_dataset_enhanced.csv'
if not os.path.exists(data_path):
    data_path = '../Data/Raw/medical_insurance_cost_dataset_enhanced.csv'
if not os.path.exists(data_path):
    data_path = r'c:\Users\Thiru kumaran v\Downloads\ML_project\Ml_Service\Data\Raw\medical_insurance_cost_dataset_enhanced.csv'

df_raw_batch = pd.read_csv(data_path).head(100).drop(columns=['annual_medical_cost_usd', 'customer_id'])

batch_preds = production_pipeline.predict(df_raw_batch)

print("==========================================================================================================")
print("                      BULK BATCH PREDICTION STATISTICS (100 SAMPLES)")
print("==========================================================================================================")
print(f"Batch Input Shape:       {df_raw_batch.shape}")
print(f"Batch Predictions Shape: {batch_preds.shape}")
print(f"Min Predicted Premium:   ${batch_preds.min():,.2f}")
print(f"Max Predicted Premium:   ${batch_preds.max():,.2f}")
print(f"Mean Predicted Premium:  ${batch_preds.mean():,.2f}")
print(f"Median Predicted Premium:${np.median(batch_preds):,.2f}")
print("Bulk Batch Inference Execution: VERIFIED SUCCESSFUL!")


                      BULK BATCH PREDICTION STATISTICS (100 SAMPLES)
Batch Input Shape:       (100, 20)
Batch Predictions Shape: (100,)
Min Predicted Premium:   $7,472.05
Max Predicted Premium:   $38,758.35
Mean Predicted Premium:  $18,037.95
Median Predicted Premium:$16,583.43
Bulk Batch Inference Execution: VERIFIED SUCCESSFUL!


## 8. Requirements Compliance Verification Matrix

| # | User Requirement | Implementation Strategy | Verification Status |
| :--- | :--- | :--- | :--- |
| **1** | Model loads successfully | Loaded `insurewise_model.joblib` via `joblib.load()` | Verified |
| **2** | Raw input passed directly to pipeline | Unprocessed DataFrames with raw string blood pressure passed to `pipeline.predict()` | Verified |
| **3** | Prediction is numeric | Verified return type is `float` / `np.floating` | Verified |
| **4** | Prediction is within reasonable range | Verified predictions lie within training bounds ($\$1,500 - \$55,000$) | Verified |
| **5** | Profile differences produce sensible variations | Tested 4 distinct customer profiles; confirmed Profile A < B < D < C | Verified |
| **6** | Missing/invalid input behavior understood | Tested NaN numericals, unknown categories, missing blood pressure strings | Verified |
| **7** | No manual preprocessing outside pipeline | End-to-end `Pipeline` handles blood pressure splitting, scaling, encoding | Verified |
| **8** | Do not retrain model | Evaluated exact saved production model artifact without retraining | Verified |

---
> **Final Conclusion:** Notebook `11_model_sanity_test.ipynb` execution is fully verified and reproducible. All sanity checks, profile sensitivity tests, edge-case handle checks, and zero-external-preprocessing requirements pass with 100% compliance. The production model artifact `models/production/insurewise_model.joblib` is fully certified for FastAPI production deployment.